# Surfaces: Jacobians, Hessians, normals, and curvature

A parametrized surface $r(u,v)\in\mathbb R^3$ has a **3×2 Jacobian** and
a **2×2 Hessian for each coordinate**. Obtain them with `dual2` and `dual4`,
then construct the fundamental forms and Gaussian/mean curvature using NumPy.

The example is a torus. Its outer and inner sides have opposite signs of
Gaussian curvature. The final interactive surface is colored by curvature
computed from dual-number derivatives at every mesh point.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from algpy.algebras import dual2, dual4
from _plotting import slider_controls, vector3

init_notebook_mode(connected=False)
pio.templates.default = "plotly_white"
major_radius, tube_radius = 2.0, .6

def surface(u, v):
    radial = major_radius + tube_radius*np.cos(v)
    return radial*np.cos(u), radial*np.sin(u), tube_radius*np.sin(v)

## First order: two independent parameter directions

Seed $u+\varepsilon_0$ and $v+\varepsilon_1$. The two generator
coefficients of each coordinate are the columns $r_u,r_v$ of the Jacobian.
The cross product of those columns determines a surface normal.

In [ ]:
e0, e1 = dual2.gens()

def first_order(u, v):
    values = surface(dual2(float(u)) + e0, dual2(float(v)) + e1)
    coefficients = np.array([dual2.to_float(value) for value in values])
    return coefficients[:, 0], coefficients[:, [1, 2]]

u0, v0 = .4, 1.1
position, jacobian = first_order(u0, v0)
normal = np.cross(jacobian[:, 0], jacobian[:, 1])
normal /= np.linalg.norm(normal)
radial0 = major_radius + tube_radius*np.cos(v0)
analytic_jacobian = np.array([
    [-radial0*np.sin(u0), -tube_radius*np.sin(v0)*np.cos(u0)],
    [ radial0*np.cos(u0), -tube_radius*np.sin(v0)*np.sin(u0)],
    [0, tube_radius*np.cos(v0)],
])
np.testing.assert_allclose(jacobian, analytic_jacobian, atol=2e-14)
np.testing.assert_allclose(normal @ jacobian, 0, atol=2e-14)
print("Position:", position)
print("Jacobian (3×2):\n", jacobian)
print("Outward unit normal:", normal)

## Second order: seed each parameter twice

Use four commuting, square-zero generators:
$u+\varepsilon_0+\varepsilon_1$ and $v+\varepsilon_2+\varepsilon_3$.
The following coefficient indices give the derivatives **without factorial
rescaling**, because the repeated directions use independent generators.

| Coefficient index | Generator product | Derivative |
| ---: | --- | --- |
| 0 | identity | value |
| 1 | $\varepsilon_0$ | $\partial_u$ |
| 4 | $\varepsilon_2$ | $\partial_v$ |
| 3 | $\varepsilon_0\varepsilon_1$ | $\partial_{uu}$ |
| 5 | $\varepsilon_0\varepsilon_2$ | $\partial_{uv}$ |
| 12 | $\varepsilon_2\varepsilon_3$ | $\partial_{vv}$ |

This implementation accepts scalar parameters or broadcastable parameter
arrays, so the same function differentiates an entire surface mesh.

In [ ]:
def surface_jet(u, v):
    u, v = np.broadcast_arrays(np.asarray(u, dtype=float), np.asarray(v, dtype=float))
    seed_u = np.zeros(u.shape + (dual4.N,))
    seed_v = np.zeros(v.shape + (dual4.N,))
    seed_u[..., 0], seed_v[..., 0] = u, v
    seed_u[..., 1] = seed_u[..., 2] = 1
    seed_v[..., 4] = seed_v[..., 8] = 1
    values = surface(dual4.from_float(seed_u), dual4.from_float(seed_v))
    coefficients = np.stack([dual4.to_float(value) for value in values], axis=-2)
    point = coefficients[..., 0]
    jacobian = coefficients[..., [1, 4]]
    hessian = np.empty(coefficients.shape[:-1] + (2, 2))
    hessian[..., 0, 0] = coefficients[..., 3]
    hessian[..., 0, 1] = hessian[..., 1, 0] = coefficients[..., 5]
    hessian[..., 1, 1] = coefficients[..., 12]
    return point, jacobian, hessian

point2, jacobian2, hessian = surface_jet(u0, v0)
np.testing.assert_allclose(point2, position, atol=2e-14)
np.testing.assert_allclose(jacobian2, jacobian, atol=2e-14)
analytic_hessian = np.empty((3, 2, 2))
analytic_hessian[:, 0, 0] = [-radial0*np.cos(u0), -radial0*np.sin(u0), 0]
analytic_hessian[:, 0, 1] = analytic_hessian[:, 1, 0] = [
    tube_radius*np.sin(v0)*np.sin(u0), -tube_radius*np.sin(v0)*np.cos(u0), 0]
analytic_hessian[:, 1, 1] = [-tube_radius*np.cos(v0)*np.cos(u0),
                            -tube_radius*np.cos(v0)*np.sin(u0), -tube_radius*np.sin(v0)]
np.testing.assert_allclose(hessian, analytic_hessian, atol=2e-14)
for coordinate, value in zip("xyz", hessian):
    print(f"Hessian of {coordinate}:\n{value}")

## Fundamental forms and curvature

For the outward normal $n=(r_u\times r_v)/\|r_u\times r_v\|$, form
\[
I=J^T J,\qquad II_{ij}=n\cdot r_{ij},\qquad S=I^{-1}II.
\]
The eigenvalues of $S$ are the principal curvatures. Their product is the
Gaussian curvature $K=\det S$ and their mean is $H=\operatorname{tr}(S)/2$.
Our sign convention is $S=-dn$: the outward-oriented sphere has negative
principal curvatures and negative $H$. Reversing the normal reverses $H$
but leaves $K$ unchanged.

In [ ]:
def surface_geometry(u, v):
    point, jacobian, hessian = surface_jet(u, v)
    ru, rv = jacobian[..., 0], jacobian[..., 1]
    normal = np.cross(ru, rv)
    normal /= np.linalg.norm(normal, axis=-1, keepdims=True)
    first_form = np.einsum("...ki,...kj->...ij", jacobian, jacobian)
    second_form = np.einsum("...k,...kij->...ij", normal, hessian)
    shape_operator = np.linalg.solve(first_form, second_form)
    gaussian = np.linalg.det(shape_operator)
    mean = np.trace(shape_operator, axis1=-2, axis2=-1)/2
    return point, jacobian, hessian, normal, first_form, second_form, gaussian, mean

sample = surface_geometry(u0, v0)
principal = np.linalg.eigvals(np.linalg.solve(sample[4], sample[5]))
print("First fundamental form:\n", sample[4])
print("Second fundamental form:\n", sample[5])
print("Principal curvatures:", principal, "K:", sample[6], "H:", sample[7])

## Differentiate a mesh and check analytic torus formulas

The dual-number calculation supplies all plotted values. The known torus
formulas below are independent checks, including both signs of $K$:
\[
K=\frac{\cos v}{r(R+r\cos v)},\qquad
H=-\frac12\left(\frac{\cos v}{R+r\cos v}+\frac1r\right).
\]
Here $R>r>0$ keeps the parametrization regular everywhere.

In [ ]:
U, V = np.meshgrid(np.linspace(0, 2*np.pi, 49), np.linspace(0, 2*np.pi, 37))
mesh = surface_geometry(U, V)
mesh_points, mesh_gaussian, mesh_mean = mesh[0], mesh[6], mesh[7]
expected_K = np.cos(V)/(tube_radius*(major_radius + tube_radius*np.cos(V)))
expected_H = -.5*(np.cos(V)/(major_radius + tube_radius*np.cos(V)) + 1/tube_radius)
np.testing.assert_allclose(mesh_gaussian, expected_K, rtol=3e-13, atol=3e-13)
np.testing.assert_allclose(mesh_mean, expected_H, rtol=3e-13, atol=3e-13)
assert np.min(mesh_gaussian) < 0 < np.max(mesh_gaussian)
print("Mesh:", U.shape, "Gaussian curvature range:", mesh_gaussian.min(), mesh_gaussian.max())

## Interactive local geometry

Move $v$ around a fixed meridian ($u=\pi/5$). The torus is symmetric in $u$,
so this visits all its curvature values. The lines show normalized Jacobian
columns and the outward normal. Dashed quadratic **normal sections** show
the second-order bending in the two principal tangent directions:
$r+sT_i+\tfrac12 k_i s^2 n$.

Hover over the moving point for $u,v,K,H$, and drag the scene to orbit it.

In [ ]:
parameters = np.linspace(0, 2*np.pi, 61)
states = [surface_geometry(np.pi/5, v) for v in parameters]

def surface_traces(index):
    point, jacobian, hessian, normal, first, second, gaussian, mean = states[index]
    tangent_u = jacobian[:, 0]/np.linalg.norm(jacobian[:, 0])
    tangent_v = jacobian[:, 1]/np.linalg.norm(jacobian[:, 1])
    # On this torus the parameter directions are orthogonal principal directions.
    curvature_u, curvature_v = second[0, 0]/first[0, 0], second[1, 1]/first[1, 1]
    s = np.linspace(-.45, .45, 35)
    section_u = point + s[:, None]*tangent_u + .5*curvature_u*s[:, None]**2*normal
    section_v = point + s[:, None]*tangent_v + .5*curvature_v*s[:, None]**2*normal
    return [
        go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode="markers",
                     marker=dict(color="#0f172a", size=6), name="Surface point",
                     text=[f"u={np.pi/5:.3f}<br>v={parameters[index]:.3f}<br>K={gaussian:.4f}<br>H={mean:.4f}"],
                     hovertemplate="%{text}<extra></extra>"),
        vector3(point, tangent_u, "Unit u tangent", "#2563eb"),
        vector3(point, tangent_v, "Unit v tangent", "#059669"),
        vector3(point, normal, "Outward unit normal", "#d97706"),
        go.Scatter3d(x=section_u[:, 0], y=section_u[:, 1], z=section_u[:, 2], mode="lines",
                     line=dict(color="#2563eb", width=5, dash="dash"), name="u normal section"),
        go.Scatter3d(x=section_v[:, 0], y=section_v[:, 1], z=section_v[:, 2], mode="lines",
                     line=dict(color="#059669", width=5, dash="dash"), name="v normal section"),
    ]

color_bound = float(np.max(np.abs(mesh_gaussian)))
fig = go.Figure(data=[go.Surface(
    x=mesh_points[..., 0], y=mesh_points[..., 1], z=mesh_points[..., 2],
    surfacecolor=mesh_gaussian, colorscale="RdBu_r", cmin=-color_bound, cmax=color_bound,
    colorbar=dict(title="Gaussian K", len=.5, y=.35), opacity=1,
    hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<br>K=%{surfacecolor:.3f}<extra></extra>",
)]+surface_traces(0))
fig.frames = [go.Frame(name=str(index), data=surface_traces(index), traces=[1, 2, 3, 4, 5, 6])
              for index in range(len(parameters))]
fig.update_layout(title="Dual-number Hessians reveal the Gaussian curvature of a torus", height=700,
                  scene=dict(xaxis=dict(title="x", range=[-3.2, 3.2]),
                             yaxis=dict(title="y", range=[-3.2, 3.2]),
                             zaxis=dict(title="z", range=[-3.2, 3.2]), aspectmode="cube"))
slider_controls(fig, [f"{v:.2f}" for v in parameters], prefix="v = ").show(auto_play=False)

**Try changing** the two radii while keeping $R>r>0$. To study another
regular surface, replace `surface(u, v)` and the analytic reference checks.
The Jacobian, coordinate Hessians, and fundamental-form calculations use
the same public dual-number API.

Additional independent dual generators can carry higher mixed derivatives;
the API tour uses `dual5` to extract a fifth mixed derivative.